In [ ]:
# ============================================================
# AI-BASED VOICE CONFIDENCE & COMMUNICATION SKILLS ANALYZER
# GOOGLE COLAB - ONE CELL
# ============================================================

!pip -q install gradio==5.44.1 faster-whisper

import re
import gradio as gr
from faster_whisper import WhisperModel

print("Loading model...")

model = WhisperModel(
    "tiny",
    device="cpu",
    compute_type="int8"
)

print("✅ Model ready!")


def analyze_voice(audio):

    if audio is None:
        return "⚠️ Please upload or record your voice."

    try:

        segments, info = model.transcribe(
            audio,
            beam_size=1
        )

        segments = list(segments)

        transcript = " ".join(
            s.text.strip()
            for s in segments
        ).strip()

        if not transcript:
            return "❌ No speech detected."

        duration = info.duration

        words = re.findall(
            r"\b[\w']+\b",
            transcript.lower()
        )

        total_words = len(words)

        minutes = max(
            duration / 60,
            0.01
        )

        wpm = int(
            total_words / minutes
        )

        unique_words = len(
            set(words)
        )

        vocabulary_ratio = (
            unique_words /
            max(total_words, 1)
        )

        vocabulary_score = min(
            100,
            int(vocabulary_ratio * 180)
        )

        # Speed
        if 110 <= wpm <= 160:
            speed_score = 100
        elif 90 <= wpm <= 180:
            speed_score = 85
        elif 70 <= wpm <= 200:
            speed_score = 70
        else:
            speed_score = 50

        # Sentence structure
        sentences = re.split(
            r"[.!?]+",
            transcript
        )

        sentences = [
            s.strip()
            for s in sentences
            if s.strip()
        ]

        avg_sentence_length = (
            total_words /
            max(len(sentences), 1)
        )

        if 8 <= avg_sentence_length <= 22:
            sentence_score = 95
        elif 5 <= avg_sentence_length <= 30:
            sentence_score = 80
        else:
            sentence_score = 60

        # Confidence estimation
        confidence = int(
            (
                speed_score +
                vocabulary_score +
                sentence_score
            ) / 3
        )

        if confidence >= 85:
            level = "High Confidence 🟢"
        elif confidence >= 70:
            level = "Moderate Confidence 🟡"
        else:
            level = "Needs Improvement 🔴"

        result = f"""
# 🎙️ Voice Communication Analysis

## 🏆 Communication Score

# {confidence}/100

### Confidence Level

**{level}**

---

## 📝 Transcript

{transcript}

---

## 📊 Analysis

| Metric | Result |
|---|---|
| Duration | {duration:.1f} seconds |
| Total Words | {total_words} |
| Speaking Speed | {wpm} WPM |
| Unique Words | {unique_words} |
| Vocabulary Score | {vocabulary_score}/100 |
| Speed Score | {speed_score}/100 |
| Sentence Score | {sentence_score}/100 |
| Overall Score | {confidence}/100 |

---

## 💡 Communication Suggestions

"""

        if wpm < 90:
            result += "- Try speaking a little faster.\\n"
        elif wpm > 180:
            result += "- Slow down to improve clarity.\\n"
        else:
            result += "- Your speaking speed is good.\\n"

        if vocabulary_score < 60:
            result += "- Improve vocabulary variety.\\n"
        else:
            result += "- Good vocabulary diversity.\\n"

        if sentence_score < 70:
            result += "- Try using clearer and shorter sentences.\\n"
        else:
            result += "- Your sentence structure is clear.\\n"

        return result

    except Exception as e:

        return f"❌ Error: {e}"


with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Voice Confidence Analyzer"
) as app:

    gr.Markdown("""
# 🎙️ AI-Based Voice Confidence & Communication Skills Analyzer

### Improve your communication by analyzing your speech.

**Voice → Speech-to-Text → NLP → Communication Analysis → Score**
""")

    audio = gr.Audio(
        sources=["upload", "microphone"],
        type="filepath",
        label="🎙️ Your Voice"
    )

    analyze = gr.Button(
        "🔍 Analyze Communication",
        variant="primary"
    )

    output = gr.Markdown()

    analyze.click(
        analyze_voice,
        inputs=audio,
        outputs=output
    )

    gr.Markdown("""
---
### TSA Concepts

Speech-to-Text • NLP • Vocabulary Analysis •
Speaking Speed • Sentence Analysis • Communication Scoring
""")


print("🚀 Launching...")

app.launch(
    share=True,
    debug=True
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.6/324.6 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/75.5M [00:00<?, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

✅ Model ready!
🚀 Launching...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8fb61274f452e06503.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
